# Inventory Optimization

## Objective

Generate inventory recommendations based on the forecasted demand to support stock planning and reduce overstocking or stockouts.

In [5]:
import pandas as pd

In [6]:
forecast = pd.read_csv(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\reports\forecast_results.csv"
)

forecast.head()

,item_id,store_id,forecast_sales
0,HOUSEHOLD_2_255,CA_1,0.453138
1,HOUSEHOLD_1_277,CA_1,0.140456
2,FOODS_3_179,CA_1,1.253311
3,HOUSEHOLD_2_242,CA_1,0.131593
4,HOUSEHOLD_2_008,CA_1,0.555120


In [8]:
def recommend_stock(sales):
    if sales >= 5:
        return "Increase Stock"
    elif sales >= 2:
        return "Maintain Stock"
    else:
        return "Reduce Stock"

forecast["Recommendation"] = forecast["forecast_sales"].apply(recommend_stock)

forecast.head(20)

,item_id,store_id,forecast_sales,Recommendation
0,HOUSEHOLD_2_255,CA_1,0.453138,Reduce Stock
1,HOUSEHOLD_1_277,CA_1,0.140456,Reduce Stock
2,FOODS_3_179,CA_1,1.253311,Reduce Stock
3,HOUSEHOLD_2_242,CA_1,0.131593,Reduce Stock
4,HOUSEHOLD_2_008,CA_1,0.555120,Reduce Stock
5,FOODS_2_046,CA_1,0.620146,Reduce Stock
6,FOODS_2_034,CA_1,0.941010,Reduce Stock
7,HOBBIES_1_273,CA_1,0.470322,Reduce Stock
8,HOBBIES_2_085,CA_1,0.500037,Reduce Stock
9,FOODS_3_149,CA_1,0.843183,Reduce Stock


In [9]:
forecast["Recommendation"].value_counts()

Recommendation
Reduce Stock      2334
Maintain Stock     467
Increase Stock     248
Name: count, dtype: int64

In [10]:
forecast.to_csv(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\reports\inventory_recommendations.csv",
    index=False
)

print("Inventory recommendations saved successfully!")

Inventory recommendations saved successfully!


## Conclusion

Based on the forecasted demand, products were categorized into three inventory actions:

- **Increase Stock:** High predicted demand.
- **Maintain Stock:** Stable predicted demand.
- **Reduce Stock:** Low predicted demand.

These recommendations can help optimize inventory levels and support data-driven stock management decisions.

In [14]:
import polars as pl
import pandas as pd

# Load processed features
data = pl.read_parquet(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\data\processed\features.parquet"
)

df = data.to_pandas()

# Keep the latest record for each product
latest_info = (
    df.sort_values("date")
      .groupby(["item_id", "store_id"])
      .tail(1)
      .reset_index(drop=True)
)

# Load inventory recommendations
inventory = pd.read_csv(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\reports\inventory_recommendations.csv"
)

# Merge with product information
dashboard_data = inventory.merge(
    latest_info[
        [
            "item_id",
            "store_id",
            "dept_id",
            "cat_id",
            "state_id",
            "date",
            "sell_price"
        ]
    ],
    on=["item_id", "store_id"],
    how="left"
)

# Reorder columns
dashboard_data = dashboard_data[
    [
        "item_id",
        "cat_id",
        "dept_id",
        "store_id",
        "state_id",
        "date",
        "sell_price",
        "forecast_sales",
        "Recommendation"
    ]
]

# Save for Power BI
dashboard_data.to_csv(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\reports\dashboard_data.csv",
    index=False
)

print("Dashboard dataset created successfully!")
dashboard_data.head()

Dashboard dataset created successfully!


,item_id,cat_id,dept_id,store_id,state_id,date,sell_price,forecast_sales,Recommendation
0,HOUSEHOLD_2_255,HOUSEHOLD,HOUSEHOLD_2,CA_1,CA,2016-05-22,9.97,0.453138,Reduce Stock
1,HOUSEHOLD_1_277,HOUSEHOLD,HOUSEHOLD_1,CA_1,CA,2016-05-22,0.97,0.140456,Reduce Stock
2,FOODS_3_179,FOODS,FOODS_3,CA_1,CA,2016-05-22,4.98,1.253311,Reduce Stock
3,HOUSEHOLD_2_242,HOUSEHOLD,HOUSEHOLD_2,CA_1,CA,2016-05-22,12.97,0.131593,Reduce Stock
4,HOUSEHOLD_2_008,HOUSEHOLD,HOUSEHOLD_2,CA_1,CA,2016-05-22,2.46,0.555120,Reduce Stock
